# 07 — Train DaViT-ALL (All-View Generalist)

Trains the generalist model on all views (CC + MLO) plus INbreast data.

**Paper hyperparameters (differ from CC/MLO):**
- head: two_layer (LayerNorm → Dropout(0.5) → Linear(1024→512) → GELU → Dropout(0.25) → Linear(512→2))
- optimizer: AdamW, lr=5e-5, betas=(0.9, 0.999), eps=1e-8, wd=1e-2
- scheduler: CosineAnnealingWarmRestarts(T_0=10, T_mult=2, eta_min=1e-7)
- include_inbreast: true (1,641 total training images)

**Important:** DaViT-ALL is excluded from INbreast evaluation to prevent data leakage.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, ConcatDataset

from multiview_davit.config import load_config
from multiview_davit.seed import set_seed
from multiview_davit.data.datasets import MedicalImageDataset
from multiview_davit.data.transforms import build_train_transform, build_inference_transform
from multiview_davit.training.multi_run import multi_run_training
from multiview_davit.evaluation.reports import plot_and_save_metrics

set_seed(42)
cfg = load_config('configs/davit_all.yaml')
data_cfg = load_config('configs/data.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Combine CBIS-DDSM train + INbreast for DaViT-ALL training
train_transform = build_train_transform(cfg.training.input_size)
val_transform   = build_inference_transform(cfg.training.input_size)

cbis_train = MedicalImageDataset(data_cfg.cbis_ddsm.train_csv, transform=train_transform)
inbreast   = MedicalImageDataset(data_cfg.inbreast.csv_path, transform=train_transform)

# Combined dataset (CBIS-DDSM + INbreast) — 1,641 training images per paper §3.4.2
combined_train = ConcatDataset([cbis_train, inbreast])
val_ds = MedicalImageDataset(data_cfg.cbis_ddsm.val_csv, transform=val_transform)

train_loader = DataLoader(combined_train, batch_size=cfg.training.batch_size,
                           shuffle=True, num_workers=cfg.training.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.training.batch_size,
                           shuffle=False, num_workers=cfg.training.num_workers, pin_memory=True)

print(f'Train: {len(combined_train)} | Val: {len(val_ds)}')

In [ ]:
best_model, history, best_score = multi_run_training(
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    checkpoint_path='checkpoints/davit_all_best.pth',
)
print(f'Best composite score: {best_score:.4f}')
plot_and_save_metrics(history, save_dir='results/', case_name='davit_all')